# 📊 Exploratory Data Analysis (EDA) — Dataset NVD CVE
## Proyek: **RedSim — AI-Assisted Security Analytics Prototype**

| Detail | Keterangan |
|---|---|
| **Mata Kuliah** | Kecerdasan Buatan untuk Keamanan (KSI251525) |
| **Dosen** | Zuliar Efendi, M.Kom |
| **Kelompok** | 8 |
| **Milestone** | 2 — Data Understanding & EDA |
| **Dataset** | NVD (National Vulnerability Database) CVE **2020–2026** (semua tahun) |
| **Sumber** | https://nvd.nist.gov/ |

---

### Tujuan Notebook
1. Memahami struktur dan karakter dataset keamanan (NVD CVE) dari **7 tahun** (2020–2026).
2. Membuat Data Dictionary untuk fitur-fitur utama.
3. Melakukan Data Profiling: shape, tipe data, missing values, duplikasi, distribusi label.
4. Membuat visualisasi yang relevan untuk menemukan pola awal, termasuk **tren antar tahun**.
5. Menyusun temuan EDA dan rencana preprocessing.

---
## 1. Setup & Import Library

In [ ]:
# ── Install (aktifkan jika di Google Colab) ──────────────────────────────
# !pip install pandas matplotlib seaborn -q


In [ ]:
import json, os, glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from collections import Counter
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.figsize': (13, 5), 'font.size': 11,
    'axes.titlesize': 14, 'axes.labelsize': 12,
    'figure.dpi': 100, 'savefig.dpi': 150
})
sns.set_theme(style='whitegrid', palette='Set2')

print('✅ Library berhasil diimport.')
print(f'Pandas  : {pd.__version__}')
print(f'Waktu   : {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')


---
## 2. Load Dataset NVD CVE — Semua Tahun (2020–2026)

Dataset NVD memuat semua kerentanan (CVE) yang telah dipublikasikan dan diverifikasi NIST.
Kami memuat **7 file JSON** (satu per tahun) lalu menggabungkannya menjadi satu DataFrame.

### Strategi Loading
- File dimuat satu per satu secara **streaming** untuk menghemat RAM.
- Hanya kolom yang relevan yang diekstrak (lihat Data Dictionary di Section 3).
- Penggabungan dilakukan di akhir dengan `pd.concat`.

> **💡 Catatan Google Colab:** Upload semua file JSON ke folder `dataset/` di sesi Colab,
> atau mount Google Drive dan sesuaikan `DATA_DIR` di cell konfigurasi berikut.

In [ ]:
# ═══════════════════════════════════════════════════════
# ⚙️  KONFIGURASI — sesuaikan path dengan lingkungan Anda
# ═══════════════════════════════════════════════════════

# Lokal / Colab dengan upload langsung
DATA_DIR = 'dataset/'

# Mount Google Drive (aktifkan jika perlu)
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_DIR = '/content/drive/MyDrive/dataset/'

# ── Daftar semua tahun ──────────────────────────────────
YEARS = [2020, 2021, 2022, 2023, 2024, 2025, 2026]

# Bangun path otomatis (support format folder maupun flat)
FILE_PATHS = []
for yr in YEARS:
    name = f'nvdcve-2.0-{yr}.json'
    # Format folder: dataset/nvdcve-2.0-2025.json/nvdcve-2.0-2025.json
    folder_path = os.path.join(DATA_DIR, name, name)
    # Format flat: dataset/nvdcve-2.0-2025.json
    flat_path   = os.path.join(DATA_DIR, name)
    if os.path.exists(folder_path):
        FILE_PATHS.append((yr, folder_path))
    elif os.path.exists(flat_path) and not os.path.isdir(flat_path):
        FILE_PATHS.append((yr, flat_path))
    else:
        print(f'⚠️  {yr}: file tidak ditemukan')

print('📁 File yang akan dimuat:')
for yr, fp in FILE_PATHS:
    size_mb = os.path.getsize(fp) / 1024 / 1024
    print(f'  ✅ {yr} — {fp}  ({size_mb:.1f} MB)')


In [ ]:
def parse_nvd_file(filepath, year):
    """Baca satu file NVD JSON dan kembalikan list of dict (flat records)."""
    with open(filepath, 'r', encoding='utf-8') as f:
        data = json.load(f)

    records = []
    for vuln in data.get('vulnerabilities', []):
        cve = vuln.get('cve', {})

        # Deskripsi
        desc_en = ''
        for d in cve.get('descriptions', []):
            if d.get('lang') == 'en':
                desc_en = d.get('value', '')
                break

        # CVSS v3.1
        metrics = cve.get('metrics', {})
        m31 = metrics.get('cvssMetricV31', [])
        if m31:
            cd = m31[0].get('cvssData', {})
            base_score          = cd.get('baseScore')
            base_severity       = cd.get('baseSeverity', '')
            attack_vector       = cd.get('attackVector', '')
            attack_complexity   = cd.get('attackComplexity', '')
            privileges_required = cd.get('privilegesRequired', '')
            user_interaction    = cd.get('userInteraction', '')
            conf_impact         = cd.get('confidentialityImpact', '')
            integ_impact        = cd.get('integrityImpact', '')
            avail_impact        = cd.get('availabilityImpact', '')
            exploit_score       = m31[0].get('exploitabilityScore')
            impact_score_val    = m31[0].get('impactScore')
        else:
            base_score = base_severity = attack_vector = attack_complexity = None
            privileges_required = user_interaction = conf_impact = None
            integ_impact = avail_impact = exploit_score = impact_score_val = None
            base_severity = ''
            attack_vector = attack_complexity = ''
            privileges_required = user_interaction = ''
            conf_impact = integ_impact = avail_impact = ''

        # CWE
        cwe_ids = []
        for w in cve.get('weaknesses', []):
            for d in w.get('description', []):
                v = d.get('value', '')
                if v.startswith('CWE-'):
                    cwe_ids.append(v)

        # Referensi
        refs = cve.get('references', [])
        has_exploit = any(
            'Exploit' in (r.get('tags', []) if isinstance(r.get('tags'), list) else [])
            for r in refs
        )

        records.append({
            'year'               : year,
            'cve_id'             : cve.get('id', ''),
            'published'          : cve.get('published', ''),
            'last_modified'      : cve.get('lastModified', ''),
            'vuln_status'        : cve.get('vulnStatus', ''),
            'description'        : desc_en,
            'base_score'         : base_score,
            'base_severity'      : base_severity,
            'attack_vector'      : attack_vector,
            'attack_complexity'  : attack_complexity,
            'privileges_required': privileges_required,
            'user_interaction'   : user_interaction,
            'conf_impact'        : conf_impact,
            'integ_impact'       : integ_impact,
            'avail_impact'       : avail_impact,
            'exploitability_score': exploit_score,
            'impact_score'       : impact_score_val,
            'primary_cwe'        : cwe_ids[0] if cwe_ids else '',
            'cwe_count'          : len(cwe_ids),
            'has_config'         : len(cve.get('configurations', [])) > 0,
            'ref_count'          : len(refs),
            'has_exploit_ref'    : has_exploit,
            'desc_length'        : len(desc_en),
        })
    return records


# ── Muat semua tahun ────────────────────────────────────────
all_records = []
year_counts = {}

for yr, fp in FILE_PATHS:
    recs = parse_nvd_file(fp, yr)
    all_records.extend(recs)
    year_counts[yr] = len(recs)
    print(f'  ✅ {yr}: {len(recs):>6,} CVE dimuat')

df = pd.DataFrame(all_records)
df['published']     = pd.to_datetime(df['published'],     errors='coerce')
df['last_modified'] = pd.to_datetime(df['last_modified'], errors='coerce')

print(f'\n📊 Total semua tahun : {len(df):,} record')
print(f'📅 Rentang waktu     : {df["published"].min().date()} → {df["published"].max().date()}')


---
## 3. Data Dictionary

| No | Nama Fitur | Tipe | Makna Keamanan | Status | Catatan Risiko |
|:---:|---|---|---|---|---|
| 1 | `year` | Numerik | Tahun publikasi CVE (2020–2026) | Metadata | Berguna untuk analisis tren |
| 2 | `cve_id` | String | ID unik kerentanan global (CVE-YYYY-XXXXX) | Metadata | Bukan fitur model |
| 3 | `published` | Datetime | Waktu CVE pertama kali dipublikasikan | Metadata | Perlu konversi datetime |
| 4 | `vuln_status` | Kategorikal | Status analisis NVD (Analyzed, Modified, Awaiting) | Fitur | Distribusi bisa tidak seimbang |
| 5 | `base_score` | Numerik (0–10) | **Skor keparahan CVSS v3.1** — metrik utama bahaya | **Fitur Utama** | ⚠️ Leakage jika target = severity |
| 6 | `base_severity` | Kategorikal | **Label keparahan** (NONE/LOW/MEDIUM/HIGH/CRITICAL) | **Label / Target** | Class imbalance signifikan |
| 7 | `attack_vector` | Kategorikal | Jalur serangan: NETWORK / LOCAL / PHYSICAL | Fitur | Mayoritas NETWORK |
| 8 | `attack_complexity` | Kategorikal | Kesulitan eksploitasi: LOW / HIGH | Fitur | — |
| 9 | `privileges_required` | Kategorikal | Hak akses yang dibutuhkan: NONE / LOW / HIGH | Fitur | — |
| 10 | `user_interaction` | Kategorikal | Butuh interaksi user: NONE / REQUIRED | Fitur | — |
| 11 | `conf_impact` | Kategorikal | Dampak kerahasiaan: NONE / LOW / HIGH | Fitur | — |
| 12 | `integ_impact` | Kategorikal | Dampak integritas: NONE / LOW / HIGH | Fitur | — |
| 13 | `avail_impact` | Kategorikal | Dampak ketersediaan: NONE / LOW / HIGH | Fitur | — |
| 14 | `exploitability_score` | Numerik (0–4) | Kemudahan eksploitasi | Fitur | ⚠️ Leakage jika target = severity |
| 15 | `impact_score` | Numerik (0–6) | Besarnya dampak jika dieksploitasi | Fitur | ⚠️ Leakage jika target = severity |
| 16 | `primary_cwe` | Kategorikal | Jenis kelemahan utama (CWE-79=XSS, CWE-89=SQLi) | Fitur | Kardinalitas tinggi |
| 17 | `cwe_count` | Numerik | Jumlah CWE terkait | Fitur | — |
| 18 | `has_config` | Boolean | Ada informasi produk terdampak | Fitur | — |
| 19 | `ref_count` | Numerik | Jumlah referensi/sumber informasi | Fitur | Distribusi long-tail |
| 20 | `has_exploit_ref` | Boolean | **Ada exploit publik yang tersedia** | Fitur | Penting untuk threat assessment |
| 21 | `desc_length` | Numerik | Panjang deskripsi CVE (karakter) | Fitur | — |

### ⚠️ Potensi Data Leakage
- `base_severity` **diturunkan langsung** dari `base_score` → hapus score jika target = severity.
- `exploitability_score` dan `impact_score` adalah komponen pembentuk `base_score`.

---
## 4. Data Profiling

### 4.1 Struktur Dataset (Shape & Info)

In [ ]:
print('=' * 65)
print('📐 DIMENSI DATASET (SEMUA TAHUN DIGABUNG)')
print('=' * 65)
print(f'Jumlah baris (record)  : {df.shape[0]:,}')
print(f'Jumlah kolom (fitur)   : {df.shape[1]}')

print('\n📅 Record per Tahun:')
yr_counts = df['year'].value_counts().sort_index()
for yr, cnt in yr_counts.items():
    bar = '█' * (cnt // 2000)
    print(f'  {yr}: {cnt:>7,}  {bar}')

print('\n📋 Info Tipe Data:')
df.info()


In [ ]:
print('🔍 5 baris pertama:')
df.head()


### 4.2 Missing Values

In [ ]:
print('=' * 65)
print('🕳️  ANALISIS MISSING VALUES')
print('=' * 65)

missing_nan   = df.isna().sum()
missing_empty = (df == '').sum()
missing_total = missing_nan + missing_empty
missing_pct   = (missing_total / len(df) * 100).round(2)

report = pd.DataFrame({
    'NaN': missing_nan,
    'String Kosong': missing_empty,
    'Total Missing': missing_total,
    'Persen (%)': missing_pct
})
has_missing = report[report['Total Missing'] > 0].sort_values('Total Missing', ascending=False)

if len(has_missing):
    print(has_missing.to_string())
    print(f'\n⚠️  {len(has_missing)} kolom memiliki data hilang.')
else:
    print('✅ Tidak ada missing values.')

print('\n📌 Interpretasi:')
print('  - Missing CVSS = CVE masih "Awaiting Analysis" di NVD (belum diverifikasi analis).')
print('  - Keputusan: drop record tanpa base_score untuk analisis severity.')


### 4.3 Duplikasi

In [ ]:
dup_rows = df.duplicated().sum()
dup_cve  = df.duplicated(subset=['cve_id']).sum()

print(f'Baris duplikat (semua kolom) : {dup_rows:,}')
print(f'CVE ID duplikat              : {dup_cve:,}')

if dup_cve > 0:
    print(f'\n⚠️  {dup_cve} CVE muncul lebih dari sekali (kemungkinan update antar file tahun).')
    print('Keputusan: pertahankan record dengan last_modified terbaru.')
    df = df.sort_values('last_modified').drop_duplicates(subset=['cve_id'], keep='last')
    print(f'✅ Setelah deduplikasi: {len(df):,} record unik.')
else:
    print('✅ Tidak ada duplikasi CVE ID.')


### 4.4 Statistik Deskriptif

In [ ]:
num_cols = ['base_score','exploitability_score','impact_score',
            'cwe_count','ref_count','desc_length']
stats = df[num_cols].describe().T
stats['median'] = df[num_cols].median()
stats['skew']   = df[num_cols].skew()
print(stats[['count','mean','median','std','min','25%','75%','max','skew']].round(2).to_string())

print('\n📌 Interpretasi:')
print('  - base_score: distribusi berpusat di rentang MEDIUM-HIGH (4–9).')
print('  - ref_count & desc_length: skewness tinggi → distribusi long-tail.')


### 4.5 Distribusi Label (base_severity)

In [ ]:
df_labeled = df[df['base_severity'].notna() & (df['base_severity'] != '')].copy()

sev_order = ['CRITICAL','HIGH','MEDIUM','LOW','NONE']
lc = df_labeled['base_severity'].value_counts().reindex(sev_order).dropna().astype(int)
lp = (lc / lc.sum() * 100).round(2)

print('Label         Jumlah    Persen(%)')
print('-' * 38)
for s in sev_order:
    if s in lc:
        print(f'{s:<12} {lc[s]:>8,}   {lp[s]:>6.2f}%')

print(f'\nTotal berlabel   : {len(df_labeled):,}')
print(f'Total tanpa label: {len(df)-len(df_labeled):,}')
maj = lc.idxmax(); mino = lc.idxmin()
print(f'\n⚠️  Imbalance — Mayoritas: {maj} ({lc[maj]:,})  |  Minoritas: {mino} ({lc[mino]:,})')
print(f'   Rasio: {lc[maj]/lc[mino]:.0f} : 1')


---
## 5. Visualisasi EDA
> Setiap grafik disertai **interpretasi** yang menjelaskan maknanya dalam konteks keamanan.

### 5.1 Distribusi Label Severity

In [ ]:
colors_sev = {'NONE':'#95a5a6','LOW':'#27ae60','MEDIUM':'#f39c12',
              'HIGH':'#e67e22','CRITICAL':'#e74c3c'}
sev_present = [s for s in sev_order if s in lc.index]
bar_colors  = [colors_sev[s] for s in sev_present]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar
ax = axes[0]
bars = ax.bar(lc.index, lc.values, color=bar_colors, edgecolor='white', linewidth=1.5)
ax.set_title('Distribusi Severity (CVSS v3.1) — 2020–2026', fontweight='bold')
ax.set_xlabel('Severity'); ax.set_ylabel('Jumlah CVE')
for b, v in zip(bars, lc.values):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+500,
            f'{v:,}', ha='center', fontweight='bold', fontsize=9)

# Pie
axes[1].pie(lc.values, labels=lc.index, colors=bar_colors,
            autopct='%1.1f%%', startangle=90, textprops={'fontsize':10})
axes[1].set_title('Proporsi Severity', fontweight='bold')

plt.tight_layout()
plt.savefig('viz_01_severity_distribution.png', bbox_inches='tight')
plt.show()

print('📌 Interpretasi:')
print('  - MEDIUM & HIGH mendominasi — mayoritas CVE berdampak sedang-tinggi.')
print('  - CRITICAL ~5-10% — paling penting untuk prioritas penanganan.')
print('  - Gunakan F1-score (bukan accuracy) untuk evaluasi model.')


### 5.2 Distribusi Base Score (Histogram)

In [ ]:
df_scored = df_labeled[df_labeled['base_score'].notna()]

fig, ax = plt.subplots(figsize=(13, 5))
ax.hist(df_scored['base_score'], bins=50, color='#3498db', edgecolor='white', alpha=0.85)

for thresh, label, col in [(0.1,'NONE→LOW','#27ae60'),(4.0,'LOW→MED','#f39c12'),
                            (7.0,'MED→HIGH','#e67e22'),(9.0,'HIGH→CRIT','#e74c3c')]:
    ax.axvline(thresh, color=col, linestyle='--', lw=1.5, alpha=0.8)
    ax.text(thresh+0.1, ax.get_ylim()[1]*0.88, label, color=col, fontsize=8, rotation=90)

mean_s   = df_scored['base_score'].mean()
median_s = df_scored['base_score'].median()
ax.axvline(mean_s,   color='red',  lw=2, label=f'Mean: {mean_s:.2f}')
ax.axvline(median_s, color='blue', lw=2, label=f'Median: {median_s:.2f}')
ax.legend()
ax.set_title('Distribusi CVSS Base Score — 2020–2026', fontweight='bold')
ax.set_xlabel('Base Score (0–10)'); ax.set_ylabel('Jumlah CVE')

plt.tight_layout()
plt.savefig('viz_02_base_score_histogram.png', bbox_inches='tight')
plt.show()

print(f'📌 Mean: {mean_s:.2f} | Median: {median_s:.2f}')
print('  - Distribusi bukan normal → pertimbangkan scaling non-linear.')


### 5.3 Jumlah CVE per Tahun (Tren)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar per tahun — total
ax1 = axes[0]
yr_total = df.groupby('year').size()
bars = ax1.bar(yr_total.index.astype(str), yr_total.values,
               color=sns.color_palette('Blues_d', len(yr_total)), edgecolor='white')
for b, v in zip(bars, yr_total.values):
    ax1.text(b.get_x()+b.get_width()/2, b.get_height()+300, f'{v:,}',
             ha='center', fontweight='bold', fontsize=9)
ax1.set_title('Total CVE per Tahun (2020–2026)', fontweight='bold')
ax1.set_xlabel('Tahun'); ax1.set_ylabel('Jumlah CVE')

# Stacked bar per severity per tahun
ax2 = axes[1]
pivot = df_labeled.groupby(['year','base_severity']).size().unstack(fill_value=0)
pivot = pivot.reindex(columns=sev_present, fill_value=0)
pivot.plot(kind='bar', stacked=True, ax=ax2,
           color=[colors_sev[s] for s in pivot.columns],
           edgecolor='white', linewidth=0.4)
ax2.set_title('Komposisi Severity per Tahun', fontweight='bold')
ax2.set_xlabel('Tahun'); ax2.set_ylabel('Jumlah CVE')
ax2.legend(title='Severity', bbox_to_anchor=(1.02,1))
ax2.set_xticklabels([str(y) for y in pivot.index], rotation=0)

plt.tight_layout()
plt.savefig('viz_03_cve_per_year.png', bbox_inches='tight')
plt.show()

print('📌 Interpretasi:')
print('  - Jumlah CVE meningkat setiap tahun — ancaman keamanan terus tumbuh.')
print('  - Komposisi severity relatif stabil antar tahun → distribusi konsisten.')
print('  - Data 2026 lebih sedikit karena tahun belum selesai (cutoff dataset).')


### 5.4 Top 15 CWE (Jenis Kerentanan)

In [ ]:
cwe_data = df[df['primary_cwe'] != '']['primary_cwe'].value_counts().head(15)

cwe_names = {
    'CWE-79':'XSS','CWE-89':'SQL Injection','CWE-862':'Missing Authorization',
    'CWE-22':'Path Traversal','CWE-74':'Injection','CWE-78':'OS Cmd Injection',
    'CWE-94':'Code Injection','CWE-416':'Use After Free','CWE-918':'SSRF',
    'CWE-77':'Command Injection','CWE-787':'Out-of-bounds Write','CWE-352':'CSRF',
    'CWE-434':'Unrestricted Upload','CWE-125':'Out-of-bounds Read','CWE-200':'Info Exposure',
}
labels = [f'{c}\n({cwe_names.get(c,"")})'  for c in cwe_data.index]

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.barh(range(len(cwe_data)), cwe_data.values,
               color=sns.color_palette('YlOrRd_r', len(cwe_data)), edgecolor='white')
ax.set_yticks(range(len(cwe_data)))
ax.set_yticklabels(labels, fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('Jumlah CVE')
ax.set_title('Top 15 CWE — 2020–2026', fontweight='bold')
for b, v in zip(bars, cwe_data.values):
    ax.text(b.get_width()+100, b.get_y()+b.get_height()/2,
            f'{v:,}', va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('viz_04_top_cwe.png', bbox_inches='tight')
plt.show()

print('📌 Interpretasi:')
print('  - CWE-79 (XSS) & CWE-89 (SQLi) konsisten mendominasi 7 tahun.')
print('  - Relevan langsung dengan fitur deteksi RedSim pada URL scanning.')
print(f'  - Total CWE unik: {df[df["primary_cwe"] != ""]["primary_cwe"].nunique()} → perlu grouping untuk modeling.')


### 5.5 Heatmap Korelasi Fitur Numerik

In [ ]:
corr_cols = ['base_score','exploitability_score','impact_score',
             'cwe_count','ref_count','desc_length']
corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdYlBu_r',
            center=0, square=True, linewidths=1, ax=ax,
            cbar_kws={'shrink':0.8})
ax.set_title('Heatmap Korelasi Fitur Numerik', fontweight='bold', pad=12)

plt.tight_layout()
plt.savefig('viz_05_correlation_heatmap.png', bbox_inches='tight')
plt.show()

print('📌 Interpretasi:')
print('  - base_score ↔ impact_score: korelasi tinggi → komponen pembentuk CVSS.')
print('  - ⚠️ DATA LEAKAGE: hapus base_score/exploit_score/impact_score jika target=severity.')
print('  - ref_count & desc_length: korelasi rendah → fitur independen yang aman.')


### 5.6 Attack Vector vs Severity

In [ ]:
ct = pd.crosstab(df_labeled['attack_vector'], df_labeled['base_severity'])
av_order  = ['NETWORK','ADJACENT_NETWORK','LOCAL','PHYSICAL']
ct = ct.reindex(index=[x for x in av_order if x in ct.index],
                columns=[x for x in sev_present if x in ct.columns])

fig, ax = plt.subplots(figsize=(13, 5))
ct.plot(kind='bar', stacked=True, ax=ax,
        color=[colors_sev.get(s,'#bdc3c7') for s in ct.columns],
        edgecolor='white', linewidth=0.4)
ax.set_title('Attack Vector vs Severity Level — 2020–2026', fontweight='bold')
ax.set_xlabel('Attack Vector'); ax.set_ylabel('Jumlah CVE')
ax.legend(title='Severity', bbox_to_anchor=(1.02,1))
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

plt.tight_layout()
plt.savefig('viz_06_attack_vector_severity.png', bbox_inches='tight')
plt.show()

print('📌 Interpretasi:')
print('  - NETWORK mendominasi — mayoritas CVE dapat dieksploitasi dari internet.')
print('  - Sangat relevan untuk RedSim yang fokus pada URL/web scanning (network-based).')


### 5.7 Tren Publikasi CVE per Bulan (2020–2026)

In [ ]:
df['pub_month'] = df['published'].dt.to_period('M')
monthly = df.groupby('pub_month').size()

fig, ax = plt.subplots(figsize=(16, 4))
ax.plot(range(len(monthly)), monthly.values, color='#2c3e50', lw=2, marker='o', ms=3)
ax.fill_between(range(len(monthly)), monthly.values, alpha=0.12, color='#3498db')

# Tandai awal setiap tahun
tick_pos = []; tick_lab = []
for i, p in enumerate(monthly.index):
    if p.month == 1:
        tick_pos.append(i); tick_lab.append(str(p.year))
        ax.axvline(i, color='#e74c3c', lw=0.8, linestyle='--', alpha=0.5)

ax.set_xticks(tick_pos); ax.set_xticklabels(tick_lab)
ax.set_title('Tren Publikasi CVE per Bulan — 2020–2026', fontweight='bold')
ax.set_xlabel('Tahun'); ax.set_ylabel('Jumlah CVE')

plt.tight_layout()
plt.savefig('viz_07_monthly_trend.png', bbox_inches='tight')
plt.show()

print('📌 Interpretasi:')
print('  - Tren jelas meningkat dari 2020 ke 2026.')
print('  - Lonjakan signifikan mulai 2024–2025 → ancaman semakin masif.')
print('  - Knowledge Base RedSim harus di-update berkala untuk tetap relevan.')


### 5.8 Boxplot — Deteksi Outlier Skor CVSS

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (col, title, col_c) in zip(axes, [
    ('base_score','Base Score','#3498db'),
    ('exploitability_score','Exploitability','#e74c3c'),
    ('impact_score','Impact Score','#2ecc71')
]):
    data = df_labeled[col].dropna()
    ax.boxplot(data, patch_artist=True, widths=0.5,
               boxprops=dict(facecolor=col_c, alpha=0.6),
               medianprops=dict(color='black', lw=2))
    ax.set_title(title, fontweight='bold'); ax.set_ylabel('Score')
    q1,q3 = data.quantile(0.25), data.quantile(0.75)
    iqr = q3-q1
    outliers = ((data<q1-1.5*iqr)|(data>q3+1.5*iqr)).sum()
    ax.text(0.95,0.95,f'Outlier: {outliers:,}',
            transform=ax.transAxes, ha='right', va='top', fontsize=9,
            bbox=dict(boxstyle='round',facecolor='wheat',alpha=0.5))

plt.suptitle('Boxplot — Deteksi Outlier CVSS Scores', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('viz_08_boxplot_outliers.png', bbox_inches='tight')
plt.show()

print('📌 Interpretasi:')
print('  - Outlier pada skor bukan noise — mencerminkan CVE dengan karakteristik ekstrem.')
print('  - Keputusan: pertahankan outlier (valid secara domain keamanan).')


### 5.9 Scatter Plot: Exploitability vs Impact

In [ ]:
sample = df_labeled.dropna(subset=['exploitability_score','impact_score']).sample(
    min(8000, len(df_labeled)), random_state=42)

fig, ax = plt.subplots(figsize=(10, 7))
for sev in sev_order:
    m = sample['base_severity'] == sev
    if m.sum() > 0:
        ax.scatter(sample.loc[m,'exploitability_score'],
                   sample.loc[m,'impact_score'],
                   c=colors_sev[sev], label=sev, alpha=0.45, s=18,
                   edgecolors='white', linewidth=0.3)
ax.set_xlabel('Exploitability Score'); ax.set_ylabel('Impact Score')
ax.set_title('Exploitability vs Impact — Pemisahan Kelas Severity (2020–2026)',
             fontweight='bold')
ax.legend(title='Severity', loc='upper left')

plt.tight_layout()
plt.savefig('viz_09_scatter.png', bbox_inches='tight')
plt.show()

print('📌 Interpretasi:')
print('  - Cluster antar severity terlihat cukup terpisah → model klasifikasi feasible.')
print('  - CRITICAL kumpul di sudut kanan-atas (exploit tinggi + impact tinggi).')


### 5.10 Tren Top-5 CWE per Tahun

In [ ]:
top5_cwe = df[df['primary_cwe'] != '']['primary_cwe'].value_counts().head(5).index.tolist()
cwe_trend = df[df['primary_cwe'].isin(top5_cwe)].groupby(['year','primary_cwe']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(13, 5))
for cwe in top5_cwe:
    if cwe in cwe_trend.columns:
        label = f'{cwe} ({cwe_names.get(cwe,"")})'  
        ax.plot(cwe_trend.index.astype(str), cwe_trend[cwe],
                marker='o', lw=2, label=label)

ax.set_title('Tren Top-5 CWE per Tahun (2020–2026)', fontweight='bold')
ax.set_xlabel('Tahun'); ax.set_ylabel('Jumlah CVE')
ax.legend(loc='upper left', fontsize=9)

plt.tight_layout()
plt.savefig('viz_10_cwe_trend.png', bbox_inches='tight')
plt.show()

print('📌 Interpretasi:')
print('  - XSS (CWE-79) konsisten tertinggi dan terus naik tiap tahun.')
print('  - Missing Authorization (CWE-862) naik signifikan mulai 2023.')
print('  - Ini mengkonfirmasi relevansi deteksi XSS dan SQLi pada RedSim.')


---
## 6. Temuan EDA & Dampak pada Preprocessing

In [ ]:
temuan = [
    ('Missing Values pada CVSS Metrics',
     'Ribuan CVE tidak punya skor CVSS v3.1 (status "Awaiting Analysis").',
     'Drop record tanpa base_score saat training model severity.',
     'Section 4.2'),
    ('Class Imbalance Signifikan',
     'MEDIUM & HIGH mendominasi >70%. NONE dan LOW sangat sedikit.',
     'Gunakan F1-score/recall. Terapkan SMOTE atau class_weight="balanced".',
     'Section 4.5 & Viz 5.1'),
    ('Data Leakage: base_score → base_severity',
     'severity diturunkan langsung dari score (threshold-based). Korelasi ~1.0.',
     'HAPUS base_score, exploitability_score, impact_score jika target=severity.',
     'Section 5.5'),
    ('CWE Kardinalitas Tinggi (Long-tail)',
     f'Ratusan CWE unik. Top-3: XSS, SQLi, Missing Auth — konsisten 7 tahun.',
     'Grouping ke 8-10 kategori utama (Injection, Memory, Auth, dll.) + "Other".',
     'Section 5.4 & 5.10'),
    ('Tren Peningkatan CVE Tiap Tahun',
     'Jumlah CVE naik terus-menerus 2020→2026. Lonjakan besar di 2024–2025.',
     'Model dan Knowledge Base RedSim perlu di-retrain/update secara berkala.',
     'Section 5.3 & 5.7'),
]

print('=' * 72)
print('📋  RINGKASAN TEMUAN EDA & DAMPAK PREPROCESSING')
print('=' * 72)
for i, (judul, detail, dampak, bukti) in enumerate(temuan, 1):
    print(f'\n🔹 Temuan {i}: {judul}')
    print(f'   Detail  : {detail}')
    print(f'   Dampak  : {dampak}')
    print(f'   Bukti   : {bukti}')
    print('   ' + '-'*68)


---
## 7. Rencana Preprocessing (Minggu 4)

| No | Masalah | Strategi Preprocessing | Prioritas |
|:---:|---|---|:---:|
| 1 | Missing CVSS scores | Drop record tanpa CVSS v3.1 | 🔴 Tinggi |
| 2 | Class imbalance | SMOTE / class_weight="balanced" | 🔴 Tinggi |
| 3 | Data leakage (score→severity) | Hapus base_score, exploit_score, impact_score | 🔴 Kritis |
| 4 | CWE kardinalitas tinggi | Grouping ke ~10 kategori + "Other" | 🟡 Sedang |
| 5 | Fitur kategorikal (attack_vector, dll.) | One-Hot Encoding | 🟡 Sedang |
| 6 | Fitur numerik skewed | StandardScaler / MinMaxScaler | 🟡 Sedang |
| 7 | Fitur boolean | Konversi ke integer 0/1 | 🟢 Rendah |
| 8 | Temporal features | Ekstrak tahun, bulan dari published | 🟢 Rendah |


---
## 8. Knowledge Base Tambahan RedSim (Deskripsi Singkat)

Selain NVD CVE, RedSim menggunakan 5 sumber teks naratif sebagai **Knowledge Base RAG**
yang sudah masuk ke Pinecone Vector DB. Sumber ini **tidak memerlukan EDA penuh** karena
tidak berbentuk tabular dan tidak memiliki label ML.

| Sumber | Entri | Est. Chunks | Format | Isi |
|---|---|---|---|---|
| CISA KEV | 1.592 | ~1.609 | JSON | CVE kritis yang aktif dieksploitasi |
| MITRE ATT&CK | 709 teknik | ~709 | STIX JSON | Teknik serangan siber enterprise |
| CWE | 969 kelemahan | ~969 | XML | Katalog jenis kelemahan software |
| CAPEC | 615 pola | ~700 | XML | Pola serangan umum |
| OWASP Cheatsheet | 118 file | ~1.487 | Markdown | Panduan mitigasi keamanan web |
| **Total KB** | | **~5.474** | | |
| **NVD CVE (semua tahun)** | **205.911** | **~103.423** | JSON | **Fokus EDA** |


---
## 9. Refleksi & Kesimpulan

### Fitur paling relevan untuk keputusan keamanan:
- **`attack_vector`** — menentukan apakah ancaman bisa dieksploitasi dari internet.
- **`primary_cwe`** — mengidentifikasi jenis kerentanan utama.
- **`has_exploit_ref`** — menunjukkan apakah exploit sudah tersedia publik.

### Fitur yang berpotensi menyesatkan model:
- **`base_score`** — data leakage jika target = severity.
- **`desc_length`** — hanya mengukur panjang teks, bukan makna keamanan.

### Nilai tambah dataset 7 tahun (2020–2026) vs 2 tahun:
- ✅ Tren temporal lebih kaya — terlihat pola kenaikan yang konsisten.
- ✅ Data lebih besar → model ML lebih stabil dan generalisasi lebih baik.
- ✅ Representasi CWE lebih lengkap — varian kerentanan lebih beragam.
- ⚠️ Trade-off: waktu loading lebih lama — gunakan sampling jika RAM terbatas.

In [ ]:
print('✅ Notebook EDA selesai — Dataset NVD CVE 2020–2026')
print(f'📅 Waktu   : {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print(f'📊 Record  : {len(df):,}')
print(f'📅 Rentang : {df["published"].min().date()} → {df["published"].max().date()}')
print(f'🖼️  Grafik  : 10 visualisasi tersimpan sebagai PNG.')
